# Stage 1/2 Notebook: Aligning Data on Reference Grid and Subtraction
make sure that the data is the same throughout

## Cell 1: setup - pull data gathered into this new notebook

In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebook" else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))

import config 

#Read fits from ztfquery data
from ztfquery.io import LOCALSOURCE
from astropy.io import fits
from astropy.wcs import WCS

#creating set-up for common grid images are going to be laid onto
from reproject import reproject_interp

import numpy as np 
import matplotlib.pyplot as plt 
import glob

#Locate images
fits_files = sorted(glob.glob(str(Path(LOCALSOURCE) / "sci" / "**" / "*sciimg.fits"), recursive=True))
print(f"Found {len(fits_files)} sciimg.fits files to align:")
for f in fits_files:
    print("  ", Path(f).name)


## Cell 2: Load each image's pixel data and WCS into memory
Cell 2 loads each image's data (a grid of brightness values -- the picture) and its wcs (a formula converting any pixel/sky coordinate, with no stars involved.)

In [ ]:
images = []

for path in fits_files:
    with fits.open(path) as hdul:
        data = hdul[0].data.astype(float)
        wcs = WCS(hdul[0].header)
    images.append((data, wcs, Path(path).name))
    print(f"Loaded {Path(path).name} shape={data.shape} WCS celestial={wcs.has_celestial}")

print(f"\nLoaded {len(images)} images into memory.")

## Cell 3: Picking the reference frame
see different images and find which one is the clearest with the least amount of blur and use that as the reference image to align the rest of the images onto

In [ ]:
# going through all images to see which one has best seeing
# lower number = better seeing
seeings = []
for path in fits_files:
    with fits.open(path) as hdul:
        seeing = hdul[0].header.get("SEEING")
    seeings.append(seeing)
    print(f"{Path(path).name} SEEING={seeing}")

# Find index of sharpest frame
ref_index = int(np.argmin(seeings))

ref_data, ref_wcs, ref_name = images[ref_index]
ref_shape = ref_data.shape

print(f"\nReference frame -> [{ref_index}] {ref_name}")
print(f"  seeing = {seeings[ref_index]} arcsec (sharpest)")
print(f"  grid   = {ref_shape}  (all images will be reprojected onto THIS)")

## Cell 4: Alignment step
Here is where we actually align all images on reference grid based on sharpest image from previous cell

In [ ]:
aligned = []

for data, wcs, name in images:
    array, footprint = reproject_interp((data, wcs), ref_wcs, shape_out=ref_shape)
    aligned.append((array, name))

    n_nan = np.isnan(array).sum()
    print(f"Aligned {name} -> NaN pixels (no coverage): {n_nan}")

print(f"\nAll {len(aligned)} images now on the reference grid {ref_shape}.")

## Cell 5 - Stack and View (see the alignment)
see the alignment image for myself in this cell block if it looks good or not

In [ ]:
stack = np.stack([a for a, _ in aligned])
mean_image = np.nanmean(stack, axis=0)

finite = mean_image[np.isfinite(mean_image)]
vmin, vmax = np.percentile(finite, [5,99])

plt.figure(figsize=(8,8))
plt.imshow(mean_image, cmap="gray", origin="lower", vmin=vmin, vmax=vmax)
plt.title(f"Mean of {len(aligned)} aligned frames", fontsize=9)
plt.colorbar(label="pixel value")
plt.show()

print("If aligned: stars are sharp points. If misaligned: stars look smeared/doubled.")

In [ ]:
# Cell 6 — VERIFY (the real Stage-1 deliverable): subtract two ALIGNED frames.
# Aligned -> static stars cancel to ~0 (flat gray noise).
# Misaligned -> each star leaves a black/white DIPOLE.

# Subtract two non-reference frames (use the reference's grid; pick frames 0 and 1).
a0, name0 = aligned[0]
a1, name1 = aligned[1]
diff = a0 - a1

# Stretch symmetrically around zero so + and - residuals are both visible.
finite = diff[np.isfinite(diff)]
lim = np.nanpercentile(np.abs(finite), 99)

plt.figure(figsize=(8, 8))
plt.imshow(diff, cmap="gray", origin="lower", vmin=-lim, vmax=+lim)
plt.title(f"Difference:\n{name0}\n - {name1}", fontsize=8)
plt.colorbar(label="pixel difference (centered on 0)")
plt.show()

print("GOOD alignment  -> stars mostly GONE; flat gray noise; few residuals.")
print("BAD alignment   -> every star is a black/white DIPOLE pair.")


## Cell 7 - Stage 2 Beginning: Save aligned image in local space

In [ ]:
from reproject import reproject_from_healpix
aligned_dir = Path(LOCALSOURCE) / "aligned"
aligned_dir.mkdir(parents=True, exist_ok=True)

ref_header = ref_wcs.to_header()

for array, name in aligned:
    out_path = aligned_dir / name
    fits.PrimaryHDU(data=array, header=ref_header).writeto(out_path, overwrite=True)
    print(f"Saved {out_path.name} ({np.isnan(array).sum()} NaN edge pixels)")

print(f"\nAll {len(aligned)} aligned frames wiritten to {aligned_dir}")